# Case study: a cleaned table of Berlin Airbnb listings with a cleaning log

*Session 4, block 3 · Author: course team · Licence: CC-BY-4.0*

The practice task of block 3: turn the findings of workbook 08 into a **documented cleaning pipeline**. Every step
is one function; every function records the rule, the number of rows affected, the action and the reason in a
**cleaning log**. The raw file is never changed. At the end, validation rules check the result, and the table
and the log are saved together.

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0). The checks behind these steps are in the
quality report of block 1 (workbook 03).

Output (not part of the repository): `case-study/data/airbnb/listings_clean.parquet` and `cleaning_log.csv` next
to it. Set the environment variable `AIRBNB_CLEAN_OUT` to write elsewhere.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.covariance import MinCovDet

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
OUT = Path(os.environ.get("AIRBNB_CLEAN_OUT", DATA / "listings_clean.parquet"))

raw = pd.read_parquet(DATA / "listings.parquet")
print(raw.shape)

log = []

def record(step, rule, affected, action, reason, df):
    log.append({"step": step, "rule": rule, "rows_affected": int(affected), "action": action,
                "reason": reason, "rows_after": len(df)})
    return df

## The steps

Each function takes a DataFrame and returns a new one; `DataFrame.pipe` chains them. Read the reason of each
step and decide whether you agree.

In [ ]:
def check_keys(df):
    dup = df["id"].duplicated()
    out = df[~dup]
    return record("1", "duplicated listing id", dup.sum(), "remove",
                  "one row per listing; a repeated id would be counted twice", out)


def fix_sentinel_max_nights(df):
    out = df.copy()
    sentinel = out["maximum_nights"].eq(2**31 - 1)
    out.loc[sentinel, "maximum_nights"] = np.nan
    return record("2", "maximum_nights = 2,147,483,647 (largest 32-bit integer)", sentinel.sum(),
                  "set to missing", "a software default, not a stay", out)


def flag_price_availability(df):
    out = df.copy()
    out["bookable"] = out["availability_365"].gt(0)
    out["price_missing"] = out["price"].isna()
    return record("3", "price missing; no free night in the next 365 days", out["price_missing"].sum(),
                  "keep missing, add indicators price_missing and bookable",
                  "structural: a listing without a bookable night shows no price; do not impute", out)


def flag_medium_term(df):
    out = df.copy()
    out["medium_term"] = out["minimum_nights"].ge(28)
    return record("4", "minimum stay of 28 nights or more", out["medium_term"].sum(), "flag medium_term",
                  "their price field is not comparable with short-stay prices (median about 23 EUR for a flat)", out)


def flag_long_minimum(df):
    out = df.copy()
    out["minimum_over_one_year"] = out["minimum_nights"].gt(365)
    return record("5", "minimum stay above 365 nights", out["minimum_over_one_year"].sum(), "flag",
                  "legal on the platform, but not a holiday rental; inspect", out)


def indicate_no_reviews(df):
    out = df.copy()
    out["no_reviews"] = out["number_of_reviews"].eq(0)
    return record("6", "review scores missing (no reviews yet)", out["no_reviews"].sum(),
                  "keep missing, add indicator no_reviews", "structural: no review, no rating", out)


def impute_bedrooms(df):
    out = df.copy()
    missing = out["bedrooms"].isna()
    rooms = missing & out["room_type"].ne("Entire home/apt")
    homes = missing & out["room_type"].eq("Entire home/apt")
    by_guests = (out[out["room_type"].eq("Entire home/apt")]
                 .groupby("accommodates")["bedrooms"].median())
    out["bedrooms_imputed"] = missing
    out.loc[rooms, "bedrooms"] = 1
    out.loc[homes, "bedrooms"] = out.loc[homes, "accommodates"].map(by_guests)
    return record("7", "bedrooms missing", missing.sum(),
                  "impute: 1 for rooms; median for the same number of guests for entire homes; flag",
                  "rooms rent one bedroom; for homes the guest rule was as good as the iterative imputer (workbook 08)", out)


def indicate_beds_bathrooms(df):
    out = df.copy()
    out["beds_missing"] = out["beds"].isna()
    out["bathrooms_missing"] = out["bathrooms"].isna()
    affected = (out["beds_missing"] | out["bathrooms_missing"]).sum()
    return record("8", "beds or bathrooms missing (about half the listings)", affected,
                  "keep missing, add indicators", "too many gaps to impute credibly; analyses that need them say so", out)


REG_NUMBER = r"^\d{2}/[A-Z]/[A-Z]{2}/\d{6}-\d{2}$"      # official format of a Berlin registration number

def check_license(df):
    out = df.copy()
    is_number = out["license_status"].eq("registration number")
    official = out["license"].astype("string").str.match(REG_NUMBER, na=False)
    out["registration_official_format"] = is_number & official
    unusual = is_number & ~official
    out = out.drop(columns="license")
    return record("9", "licence field: registration numbers not in the official format", unusual.sum(),
                  "flag; keep license_status, drop the raw field",
                  "a registration number identifies a flat; the analysis needs only the status", out)


def flag_extreme_prices(df):
    out = df.copy()
    comparable = out["price"].notna() & ~out["medium_term"]
    log_p = np.log(out["price"].where(comparable))
    med = log_p.groupby(out["room_type"]).transform("median")
    mad = (log_p - med).abs().groupby(out["room_type"]).transform("median")
    robust_z = 0.6745 * (log_p - med) / mad
    out["price_extreme"] = robust_z.abs().gt(3.5).fillna(False)
    return record("10", "short-stay price far from its room type (robust z of log price > 3.5)",
                  out["price_extreme"].sum(), "flag, do not delete",
                  "typing errors and blocking prices cannot be told apart from luxury flats without inspection", out)


def add_log_price(df):
    out = df.copy()
    out["log_price"] = np.log(out["price"])
    return record("11", "price is strongly right-skewed", out["price"].notna().sum(), "add log_price",
                  "multiplicative differences; raw column kept", out)


def flag_mahalanobis(df):
    out = df.copy()
    use = out["price"].notna() & ~out["medium_term"]
    X = np.column_stack([out.loc[use, "log_price"], np.log(out.loc[use, "accommodates"])])
    d2 = MinCovDet(random_state=0).fit(X).mahalanobis(X)
    out["mahalanobis_flag"] = False
    out.loc[use, "mahalanobis_flag"] = d2 > chi2.ppf(0.999, df=2)
    return record("12", "unusual price for the number of guests (robust Mahalanobis, chi2 0.999)",
                  out["mahalanobis_flag"].sum(), "flag", "inspect; hostels with dormitory beds are genuine", out)


clean = (raw.pipe(check_keys).pipe(fix_sentinel_max_nights).pipe(flag_price_availability)
         .pipe(flag_medium_term).pipe(flag_long_minimum).pipe(indicate_no_reviews).pipe(impute_bedrooms)
         .pipe(indicate_beds_bathrooms).pipe(check_license).pipe(flag_extreme_prices)
         .pipe(add_log_price).pipe(flag_mahalanobis))
cleaning_log = pd.DataFrame(log)
cleaning_log[["step", "rule", "rows_affected", "action"]]

## Look at what the flags found

In [ ]:
print(clean["license_status"].value_counts())
print(pd.crosstab(clean["calculated_host_listings_count"].gt(1).rename("host with several listings"),
                  clean["license_status"], normalize="index").round(3))

In [ ]:
cols = ["price", "property_type", "room_type", "accommodates", "minimum_nights", "number_of_reviews"]
print("extreme prices, highest and lowest:")
print(clean.loc[clean["price_extreme"], cols].sort_values("price").iloc[[0, 1, 2, -3, -2, -1]])
print("\nMahalanobis flags by room type:")
print(clean.loc[clean["mahalanobis_flag"], "room_type"].value_counts())

**Interpretation.** Hosts with several listings fill in the licence field more often than single-listing
hosts (81 % against 59 %), but mostly with the name of a legal entity; a registration number
is more common among single-listing hosts (46 % against 22 %). The two fields measure different things, so
"has a licence entry" and "has a registration number" must not be mixed up in a report. The extreme prices at the
top are the prices in the thousands of workbook 08: a loft and a house that are implausible for their size, but
also a houseboat for 16 guests, which may well be genuine; at the bottom are short-stay rooms offered for about
€10. The Mahalanobis step flags 94 unusual combinations of price and guests: entire homes with implausible prices,
and shared rooms with many beds at low prices (hostel dormitories, which are genuine). Flags are questions, not
verdicts.

## Validate, then save

In [ ]:
short = clean[clean["price"].notna() & ~clean["medium_term"]]
assert clean["id"].is_unique
assert not clean["maximum_nights"].eq(2**31 - 1).any()
assert clean["bedrooms"].notna().all(), "every listing has a bedroom count after step 7"
assert (clean["review_scores_rating"].isna() == clean["no_reviews"]).all()
assert "license" not in clean.columns, "the raw licence field is not saved"
assert short["price"].between(1, 20_000).all()
assert set(clean["license_status"]) <= {"missing", "registration number", "legal entity name",
                                        "private host name", "other"}
print(f"{len(clean):,} listings, {len(short):,} with a comparable short-stay price "
      f"(median {short['price'].median():.0f} EUR)")

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(OUT, index=False)
cleaning_log.to_csv(OUT.with_name("cleaning_log.csv"), index=False)
print("saved", OUT, "and", OUT.with_name("cleaning_log.csv").name)

**Exercises.**

1. Step 7 imputes bedrooms. Write the counter-argument of a colleague who would rather keep them missing, and
   decide.
2. Add a step that flags listings whose coordinates fall outside the 12 districts (hint: the bounding box of
   `neighbourhoods.geojson`). How many are there?
3. Which steps would you have to re-run if the next snapshot from Inside Airbnb arrived tomorrow? Which
   thresholds would you keep fixed, and why?